# Experiment 1 — single-turn Unicode/ASCII smuggling baseline

Proposal 4.5 **Experiment 1**: ASR of each encoding family on its own, single-turn, over the
JBB **Goal** requests. This notebook drives the run through `run_experiment.py` (the experiment
layer) so there is no code drift between interactive and headless runs. Everything is injected;
nothing is added to the `smuggle` package.

**What is measured** (saved for *every* variant): the encoded prompt, the target reply, the
judge verdict + rationale (task-achieved AND not-refused), guardrail **bypass** on the encoded
input, and imperceptibility. Outputs go to `runs/` as full transcripts (`.txt`), lossless rows
(`.jsonl`), and analysis tables (ASR + Wilson CI, bypass vs ASR, targeted-vs-random McNemar).

**Deliberate, logged deviations from the proposal baseline** (so they are reported honestly):
- **Targets** are `gpt-oss:20b-cloud`, `gemma4:31b-cloud`, `nemotron-3-nano:30b-cloud` — a 20-31B
  cloud tier chosen for stronger Unicode decoding, **not** the proposal's named 7-9B/70-72B
  models. So this run is an **exploratory** characterisation; it does not populate Objective 4
  (model scale 7-9B vs 70-72B).
- **`max_completion_tokens=4096`** (not 512): these are reasoning models and 512 is spent on
  hidden reasoning, which blanks the reply. Greedy decoding (temp=0, top_p=1) is kept.
- **Judge** = `beam:cloud` (independent of all three target model families).
- **Inference request** = temperature 0, top-p 1, `seed=0`. Ollama Cloud still varied on exact
  replays, so results are deployment observations; byte-identical inputs are collapsed before
  ASR/McNemar and all raw calls remain available.

> Run order: **Restart Kernel, then Run All** so the durable SQLite store initialises before
> `smuggle` is imported.

## 0 · Setup — durable store, roles, dataset

In [ ]:
# The durable PyRIT SQLite store MUST be chosen BEFORE smuggle is imported (proposal 4.9).
import os
os.environ["SMUGGLE_DB"] = "runs/notebook_reporting.db"   # notebook/reporting store

import run_experiment as X   # experiment layer; importing it imports smuggle with the db set
import smuggle

print("families    :", X.CORE_FAMILIES)
print("targets     :", X.TARGET_MODELS)
print("judge       :", X.JUDGE_MODEL)
print("guard       :", X.GUARD_MODEL, "| adversarial:", X.ADVERSARIAL_MODEL)
print("token budget:", X.TARGET_MAX_TOKENS, "(target/judge) | greedy temp=0.0, top_p=1.0")
print("inference seed:", X.INFERENCE_SEED, "(requested; cloud service is not byte-deterministic)")
print("JBB Goals   :", len(X.load_goals(0)), "available")

## 1 · Completed seeded pilot — 5 Goals end-to-end

The 300-variant pilot completed with 0 errors and 0 blanks. Canonical results are in
`runs/pilot_seeded/`; its observation/falsification ledger is `observation_ledger.md`. The code
below loads the completed rows without rerunning cloud requests.


In [ ]:
import json
with open("runs/pilot_seeded/exp1_pilot_seeded_rows.jsonl", encoding="utf-8") as fh:
    pilot_rows = [X._hydrate_outcome(json.loads(line)) for line in fh if line.strip()]
pilot_errors = []
print("pilot variants:", len(pilot_rows), "| errors:", len(pilot_errors))

## 2 · Inspect the pilot

Headline behaviour-level ASR + 95% CI, the bypass-vs-ASR table, and one full transcript (the same
panel `result.print()` shows: encoded prompt, reply, verdict + rationale).

In [ ]:
print(smuggle.format_report_table(smuggle.bypass_and_asr(pilot_rows)))
print()
for r in smuggle.asr_over_prompts(pilot_rows, aggregate="any"):
    print(f"{r.model[:24]:24} {r.family:20} {r.mode:9} ASR={r.asr:>5.0%} "
          f"[{r.ci_low:.2f},{r.ci_high:.2f}] n_goals={r.n_prompts}")

# One full transcript (encoded prompt + reply + verdict + rationale):
print("\n" + smuggle.format_variant(pilot_rows[0]))

## 3 · Full run — all 100 Goals

This is long (thousands of cloud calls). Prefer running it **headless** in a terminal so closing
the notebook never interrupts it; outputs land in `runs/full/`, checkpointed per target:

```bash
SMUGGLE_DB="runs/exp1_full.db" python run_experiment.py --goals 100 --out runs/full --tag exp1_full
```

To run it from the notebook instead, uncomment the call below.

In [ ]:
# full_rows, full_errors = await X.run_exp1_async(
#     goals=X.load_goals(100), out_dir="runs/full", tag="exp1_full",
# )
# print("full variants:", len(full_rows), "| errors:", len(full_errors))